In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass


EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXXX" # Add your namespace-prefix here

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.visual-transcript",
        description="Describe the given video",
        worker_release="qwen3-instruct",
        text_prompt="""
          You are given a short video segment (approximately 1–5 seconds).

          Your task is to produce a factual description of what is visually observable.

          Write EXACTLY 100 words.

          ----------------------------------------
          INSTRUCTIONS:

          1. Describe only what is visually observable.
          2. Do NOT infer intent, emotions, motivations, or off-screen events.
          3. Do NOT speculate.
          4. Do NOT add background context not visible in the frames.
          5. Do NOT mention camera quality, resolution, or metadata.
          6. Use clear, neutral, objective language.
          7. Use present tense.
          8. Avoid repetition.
          9. No bullet points.
          10. No introduction or conclusion.

          ----------------------------------------
          CONTENT GUIDELINES:

          Include when visible:
          - Subjects (people, animals, objects)
          - Actions occurring
          - Spatial relationships
          - Environment or setting
          - Notable motion
          - Visible text or numbers
          - Lighting conditions
          - Weather conditions (if visible)

          ----------------------------------------
          STRICT OUTPUT RULES:

          - Output exactly 100 words.
          - Do not include a word count.
          - Do not include commentary.
          - Do not include quotation marks unless text appears visibly in the scene.
          - If the content is unclear or visually ambiguous, describe what is clearly visible.
          - If the segment is blank or unreadable, output: NO

          ----------------------------------------

          Return only the 100-word description.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=10,
            image_size=512
        ),
        is_public=False
    )
]



In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from pathlib import Path

video_length = ... # replace with length of video

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.describe.visual-transcript:latest",
       videoChunkLengthSeconds=video_length
   )
])

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_video_path = Path("/content/sample_video.mp4")
   job = endpoint.upload(sample_video_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")